# K-Means Clustering

This model clusters the *provinces* into k-groups to the nearest randomly generated centroid using Euclidean distance.

In [1]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from methods.read_data import createMatrix, printMatrix
from methods.normalize_data import normalizeData, standardizeData, coalesceData
from models.clustering import k_means_clustering
from evaluation.conf_matrix import evaluate
from pathlib import Path
from os import path
from copy import deepcopy
import csv

### Step 1: Create the Dataset from the `PROJECT_DATASET` folder
- This will create a matrix with 12 fields/attributes and 87 provinces
- The 4 districts of NCR are used as its provinces, and Isabela City and Cotabato City are separate rows like in the poverty tables
- The internet use dataset is by region only, so the share of family spending on Information and Communication (FIES 2023 Table 7) is used instead

In [3]:
data_set_dir = path.join(path.abspath("."), "PROJECT_DATASET")          # Get the path to the dataset folder
provinces, fields, matrix = createMatrix(Path(data_set_dir))            # Create the matrix from the dataset
printMatrix(provinces, fields, matrix)                                  # Print the matrix with all of the provinces and fields

87 provinces x 12 fields
showing fields 0 to 11 

  [0] Table_2 | Poverty Incidence among Population (%) 2023
  [1] Table_2 | Poverty Incidence among Population (%) 2021
  [2] Table_2 | Poverty Incidence among Population (%) 2018
  [3] Table_12 | Estimate (%) 2023
  [4] Table_6 | Estimate ('000) 2023
  [5] Table_2 | Annual Per Capita Poverty Threshold (in PhP) 2023
  [6] Total Family Size | Average Income
  [7] Total Family Size | Average Expenditure
  [8] Table_7 | Information and Communication
  [9] DERIVED | Change in poverty incidence 2021 to 2023 (percentage points)
  [10] DERIVED | Change in poverty incidence 2018 to 2023 (percentage points)
  [11] DERIVED | Income inequality (top 10% share / bottom 40% share)

Province                       [0]         [1]         [2]         [3]         [4]         [5]         [6]         [7]         [8]         [9]        [10]        [11]
NCR 1st District               2.4         1.7         3.0         0.1       46.54     37711.0      482.79

### Step 2: Cleanup the data
- Convert the matrix into a dictionary of province statistics with province name as key and list of attributes as values
- Coalesce missing values to 0 for clustering, then standardize it using Z-score Scaling

In [4]:

# Coalesce NA values to 0 for clustering, then standardize it.
cleaned_data = standardizeData(coalesceData(matrix))

# Create dictionary of province statistics with province name as key and list of features as value
province_stats = {provinces[i]: cleaned_data[i] for i in range(len(provinces))}


### Step 3: Run the K-means clustering with k=5
- Run the k-means clustering with k set to 5
- Then display the categories for each province in the Dataset

In [5]:
copied_province_stats = deepcopy(province_stats)
categories = k_means_clustering(copied_province_stats, 5)
# For each labeled dataset, it includes the (1) label/cluster, (2) name of the province, and (3) data points associated with the province
print("\nLabeled Dataset:")
for id, points in province_stats.items():
  print(f"{categories[id]:<1} : {id:<22} {points}")

Initial Centroids
(-0.8510548968799746, -1.1886637324415386, -1.1223100691084296, -0.6046318152745008, 0.45331311408701536, 1.7530533115632254, 2.5052570266410052, 2.535429168914644, 1.4525216146035456, 0.7008826742807555, 0.6606936427092717, 0.5021876726755643)
(-0.19989012176953452, -0.3383965610342303, 1.8668961884319957, -0.7378038859805303, -0.9558653318346052, -0.917814937159845, 0.029639591205001103, -0.062242313125960104, -0.9870417149429489, 0.27178124569295004, -2.7047672936554767, -0.6227546504577995)
(-1.4588086869830519, -1.4175818170511987, -1.3408756879393424, -1.137320098098618, -0.7429282733256694, 1.366446211263408, 3.4597414960114556, 3.0357453130067795, 1.4982634270325428, 0.07942543287772692, 0.3500357101217565, -0.19710079846139156)
(0.19949094029820216, -0.27299139400289885, -0.7173208342158559, 0.5939168210797627, -0.04993436823096683, -0.5090584717386836, -0.5891679377673468, -0.5256968242842682, 0.6901580741202661, 0.834052083152833, 1.161198089655824, 0.13734

### Step 4: Evaluate the clusters using a Confusion Matrix
- Read the actual class of each province from the tagged dataset `province_policy_results.csv` (created by `methods/label_data.py` from the policies in the Policy for Labeling Data section of the project document)
- K-means only gives a cluster number, so each cluster is labeled with the most common actual class among its provinces (majority vote)
- Compare the predicted class with the actual class and compute the Recall, Precision, F1 Score and Accuracy for each class (one vs. rest)

In [6]:
# Read the actual class of each province from the tagged dataset
tagged_dir = path.join(data_set_dir, "province_policy_results.csv")    # Get the path to the tagged dataset
with open(tagged_dir, newline="", encoding="utf-8-sig") as file:
  actual_classes = {row["Province"]: row["Class"] for row in csv.DictReader(file)}

# Label each cluster with the most common actual class among its provinces (majority vote)
cluster_labels = {}
for cluster in set(categories.values()):
  cluster_classes = [actual_classes[id] for id in categories if categories[id] == cluster]
  cluster_labels[cluster] = max(cluster_classes, key=cluster_classes.count)

# Display the class assigned to each cluster
print("Cluster Labels:")
for cluster, label in sorted(cluster_labels.items()):
  print(f"{cluster:<1} : {label}")

# Create the list of actual and predicted classes for each province
actual = [actual_classes[id] for id in provinces]                     # Actual class from the tagged dataset
predicted = [cluster_labels[categories[id]] for id in provinces]      # Predicted class from the cluster label

print("\nActual vs. Predicted:")
for i in range(len(provinces)):
  print(f"{provinces[i]:<22} - {actual[i]:<9} {predicted[i]}")
print()

# Calculate and display the confusion matrix and the performance metrics for each class
results = evaluate(actual, predicted)

Cluster Labels:
0 : RETAIN
1 : RETAIN
2 : DECREASE
3 : INCREASE
4 : INCREASE

Actual vs. Predicted:
NCR 1st District       - DECREASE  DECREASE
NCR 2nd District       - DECREASE  DECREASE
NCR 3rd District       - DECREASE  RETAIN
NCR 4th District       - DECREASE  DECREASE
Abra                   - RETAIN    RETAIN
Apayao                 - RETAIN    RETAIN
Benguet                - RETAIN    RETAIN
Ifugao                 - RETAIN    RETAIN
Kalinga                - RETAIN    RETAIN
Mt. Province           - RETAIN    RETAIN
Ilocos Norte           - RETAIN    DECREASE
Ilocos Sur             - RETAIN    RETAIN
La Union               - RETAIN    RETAIN
Pangasinan             - RETAIN    INCREASE
Batanes                - RETAIN    RETAIN
Cagayan                - RETAIN    RETAIN
Isabela                - RETAIN    RETAIN
Nueva Vizcaya          - RETAIN    RETAIN
Quirino                - RETAIN    RETAIN
Aurora                 - RETAIN    RETAIN
Bataan                 - RETAIN    RETAIN
Bulacan 